In [0]:
%sql
SELECT
    COUNT(*) AS total_predictions,
    COUNT(DISTINCT UDI) AS unique_machines,
    SUM(predicted_failure) AS predicted_failures,
    MAX(prediction_timestamp) AS latest_prediction
FROM predictivemaintenancecatalog.gold_data.test_failure_predictions;

In [0]:
landing_table = "predictivemaintenancecatalog.raw_data.landing_testdata"

spark.sql(f"DESCRIBE DETAIL {landing_table}") \
    .select("location") \
    .show(truncate=False)

In [0]:
landing_table = "predictivemaintenancecatalog.raw_data.landing_testdata"

landing_path = "/Volumes/predictivemaintenancecatalog/raw_data/landing/testdata"

print("Registered Table Rows:",
      spark.table(landing_table).count())

print("Volume Delta Rows:",
      spark.read.format("delta").load(landing_path).count())

In [0]:
spark.sql("""
SELECT COUNT(*) AS new_records
FROM predictivemaintenancecatalog.raw_data.landing_testdata
WHERE UDI BETWEEN 1000501 AND 1000600
""").show()

In [0]:
%sql
SELECT *
FROM predictivemaintenancecatalog.gold_data.prediction_monitoring_drift
ORDER BY calculated_at DESC
LIMIT 20;

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT UDI) AS unique_udis,
    SUM(
        CASE WHEN Machine_failure IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_labels
FROM predictivemaintenancecatalog.gold_data.streaming
WHERE UDI BETWEEN 1000701 AND 1000800;

In [0]:
%sql
SELECT UDI, Machine_failure
FROM predictivemaintenancecatalog.gold_data.streaming
WHERE UDI BETWEEN 1000701 AND 1000800
ORDER BY UDI
LIMIT 20;

In [0]:
repair_staging_path = (
    "/Volumes/predictivemaintenancecatalog/"
    "raw_data/landing/repair_staging"
)

dbutils.fs.mkdirs(repair_staging_path)

print("Folder created successfully!")

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/predictivemaintenancecatalog/raw_data/landing/"
    )
)

In [0]:

from pyspark.sql import functions as F

REPAIR_PATH = (
    "/Volumes/predictivemaintenancecatalog/"
    "raw_data/landing/repair_staging/"
    "ai4i_csv_inbox_batch_004_repair.csv"
)

GOLD_TABLE = (
    "predictivemaintenancecatalog.gold_data.streaming"
)

# Read corrected CSV
repair_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(REPAIR_PATH)
)

# Keep only the columns needed for the repair
repair_labels = repair_df.select(
    F.col("UDI").cast("long"),
    F.col("Machine_failure").cast("long")
)

# Validate
print("Total records:", repair_labels.count())
print("Unique UDIs:", repair_labels.select("UDI").distinct().count())
print("Missing labels:", repair_labels.filter(
    F.col("Machine_failure").isNull()
).count())

print("Invalid labels:", repair_labels.filter(
    ~F.col("Machine_failure").isin(0, 1)
).count())

print("\nFailure label distribution:")
repair_labels.groupBy("Machine_failure").count().show()

# Check that the UDIs match the existing Gold records
gold_batch = (
    spark.table(GOLD_TABLE)
    .filter(F.col("UDI").between(1000701, 1000800))
)

print("Existing Gold records:", gold_batch.count())

print("\nGold table metadata:")
spark.sql(f"DESCRIBE TABLE EXTENDED {GOLD_TABLE}").show(
    100, truncate=False
)


In [0]:
GOLD_TABLE = (
    "predictivemaintenancecatalog.gold_data.streaming"
)

gold_details = spark.sql(
    f"DESCRIBE DETAIL {GOLD_TABLE}"
)

display(
    gold_details.select(
        "name",
        "format",
        "location",
        "properties"
    )
)

In [0]:
spark.table(GOLD_TABLE).printSchema()

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

GOLD_TABLE = "predictivemaintenancecatalog.gold_data.streaming"

REPAIR_PATH = (
    "/Volumes/predictivemaintenancecatalog/"
    "raw_data/landing/repair_staging/"
    "ai4i_csv_inbox_batch_004_repair.csv"
)

# 1. Read corrected labels
repair_labels = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(REPAIR_PATH)
    .select(
        F.col("UDI").cast("long"),
        F.col("Machine_failure").cast("long")
    )
)

# 2. Validate source
assert repair_labels.count() == 100
assert repair_labels.select("UDI").distinct().count() == 100
assert repair_labels.filter(
    F.col("Machine_failure").isNull()
    | ~F.col("Machine_failure").isin(0, 1)
).count() == 0

# 3. Confirm Delta format
table_format = (
    spark.sql(f"DESCRIBE DETAIL {GOLD_TABLE}")
    .select("format")
    .first()["format"]
)

assert table_format.lower() == "delta", \
    f"Expected Delta table, found {table_format}"

# 4. Validate existing Gold records
gold_batch = (
    spark.table(GOLD_TABLE)
    .filter(F.col("UDI").between(1000701, 1000800))
)

assert gold_batch.count() == 100
assert gold_batch.select("UDI").distinct().count() == 100

# Confirm exact UDI matches in both directions
source_unmatched = (
    repair_labels.select("UDI")
    .join(gold_batch.select("UDI"), "UDI", "left_anti")
    .count()
)

target_unmatched = (
    gold_batch.select("UDI")
    .join(repair_labels.select("UDI"), "UDI", "left_anti")
    .count()
)

assert source_unmatched == 0
assert target_unmatched == 0

# 5. Update existing records ONLY
(
    DeltaTable.forName(spark, GOLD_TABLE)
    .alias("target")
    .merge(
        repair_labels.alias("source"),
        "target.UDI = source.UDI"
    )
    .whenMatchedUpdate(
        set={
            "Machine_failure": "source.Machine_failure"
        }
    )
    .execute()
)

print("Batch 004 labels repaired successfully!")

# 6. Verify
display(
    spark.table(GOLD_TABLE)
    .filter(F.col("UDI").between(1000701, 1000800))
    .groupBy("Machine_failure")
    .count()
    .orderBy("Machine_failure")
)


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,

    SUM(CASE
        WHEN _quality_pass = true THEN 1 ELSE 0
    END) AS quality_passed,

    SUM(CASE
        WHEN _quality_pass = false THEN 1 ELSE 0
    END) AS quality_failed,

    SUM(CASE
        WHEN Machine_failure IS NULL THEN 1 ELSE 0
    END) AS missing_labels,

    SUM(CASE
        WHEN _quality_pass = true
         AND Machine_failure IS NULL
        THEN 1 ELSE 0
    END) AS incorrectly_passed

FROM predictivemaintenancecatalog.silver_data.streaming

WHERE UDI BETWEEN 1000701 AND 1000800;

In [0]:
SILVER_TABLE = (
    "predictivemaintenancecatalog.silver_data.streaming"
)

display(
    spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}")
    .select("name", "format", "location", "properties")
)

In [0]:
spark.sql(
    f"DESCRIBE HISTORY {SILVER_TABLE}"
).select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(10, truncate=False)

In [0]:
%sql
SELECT 'Bronze' AS layer,
       COUNT(*) AS records,
       COUNT(DISTINCT UDI) AS unique_udis
FROM predictivemaintenancecatalog.bronze_data_not_quality_checked.streaming
WHERE UDI BETWEEN 1000801 AND 1000900

UNION ALL

SELECT 'Silver',
       COUNT(*),
       COUNT(DISTINCT UDI)
FROM predictivemaintenancecatalog.silver_data.streaming
WHERE UDI BETWEEN 1000801 AND 1000900

UNION ALL

SELECT 'Gold',
       COUNT(*),
       COUNT(DISTINCT UDI)
FROM predictivemaintenancecatalog.gold_data.streaming
WHERE UDI BETWEEN 1000801 AND 1000900;

In [0]:
%sql
SELECT
    feature,
    feature_type,
    current_records,
    psi,
    ks_statistic,
    ks_pvalue,
    psi_status,
    ks_significant,
    calculated_at,
    rolling_window_size,
    minimum_udi,
    maximum_udi
FROM predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_rolling
WHERE monitoring_run_id = (
    SELECT monitoring_run_id
    FROM predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_rolling
    ORDER BY calculated_at DESC
    LIMIT 1
)
ORDER BY psi DESC;

In [0]:
%sql

SELECT
    alert_id,
    alert_run_id,
    evaluated_state,
    notified_subscriptions,
    failed_subscriptions,
    error_code,
    error_message,
    start_time,
    end_time

FROM system.alert.alert_evaluation_history

WHERE alert_id = '2582872932567394'

ORDER BY start_time DESC

LIMIT 10;


In [0]:
%sql
SELECT
    alert_id,
    display_name,
    owned_by,
    create_time,
    change_time,
    delete_time

FROM system.alert.alerts

WHERE alert_id = '2582872932567394'

ORDER BY change_time DESC;

In [0]:
%sql
SELECT
    alert_id,
    evaluated_state,
    notified_subscriptions,
    failed_subscriptions,
    error_code,
    error_message,
    start_time,
    end_time

FROM system.alert.alert_evaluation_history

ORDER BY start_time DESC

LIMIT 20;

In [0]:
%sql
SELECT
    DATE_TRUNC('HOUR', prediction_timestamp) AS prediction_hour,
    COUNT(*) AS total_predictions,
    SUM(CAST(predicted_failure AS INT)) AS predicted_failures,
    ROUND(
        100.0 * SUM(CAST(predicted_failure AS INT)) / COUNT(*),
        2
    ) AS predicted_failure_rate_pct,
    ROUND(AVG(failure_probability), 4) AS mean_probability,
    ROUND(MIN(failure_probability), 4) AS min_probability,
    ROUND(MAX(failure_probability), 4) AS max_probability

FROM predictivemaintenancecatalog.gold_data.test_failure_predictions

WHERE prediction_timestamp >= '2026-11-02'

GROUP BY 1
ORDER BY 1;

In [0]:
%sql
SELECT
    COUNT(*) AS total_predictions,
    SUM(CAST(predicted_failure AS INT)) AS predicted_failures,
    ROUND(
        100.0 * SUM(CAST(predicted_failure AS INT)) / COUNT(*),
        2
    ) AS predicted_failure_rate_pct
FROM predictivemaintenancecatalog.gold_data.test_failure_predictions;

In [0]:
%sql
SELECT
    feature,
    ROUND(psi, 4) AS psi,
    psi_status
FROM predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_rolling
WHERE calculated_at = (
    SELECT MAX(calculated_at)
    FROM predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_rolling
)
ORDER BY psi DESC;

In [0]:
%sql
SELECT *
FROM predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_alert_state
LIMIT 20;

In [0]:
%sql
SELECT *
FROM predictivemaintenancecatalog.gold_data.prediction_monitoring_drift_alert_events
LIMIT 20;

In [0]:
%sql
SELECT
    job_id,
    run_id,
    period_start_time,
    period_end_time,
    result_state
FROM system.lakeflow.job_run_timeline
WHERE period_start_time >= CURRENT_TIMESTAMP() - INTERVAL 14 DAYS
ORDER BY period_start_time DESC
LIMIT 30;